In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
# Install latest ultralytics (required for YOLO26 support)
%pip install -q -U ultralytics supervision roboflow
# Disable Ultralytics telemetry
!yolo settings sync=False
import ultralytics
ultralytics.checks()

In [ ]:
# Sanity check: run pretrained YOLO26n on a sample image
!yolo task=detect mode=predict model=yolo26n.pt conf=0.25 source='https://media.roboflow.com/notebooks/examples/dog.jpeg' save=True

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/predict/dog.jpg', width=600)

In [ ]:
from ultralytics import YOLO
from PIL import Image
import requests

model = YOLO('yolo26n.pt')
image = Image.open(requests.get('https://media.roboflow.com/notebooks/examples/dog.jpeg', stream=True).raw)
result = model.predict(image, conf=0.25)[0]

In [ ]:
result.boxes.xyxy

In [ ]:
result.boxes.conf

In [ ]:
result.boxes.cls

In [ ]:
import supervision as sv

detections = sv.Detections.from_ultralytics(result)

In [ ]:
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator(text_color=sv.Color.BLACK)

annotated_image = image.copy()
annotated_image = box_annotator.annotate(annotated_image, detections=detections)
annotated_image = label_annotator.annotate(annotated_image, detections=detections)

sv.plot_image(annotated_image, size=(10, 10))

In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets

from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

workspace = rf.workspace("renaire-odarve")
project = workspace.project("erotica-detection")
version = project.version(4)
dataset = version.download("yolov11")

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolo26n.pt data={dataset.location}/data.yaml epochs=150 imgsz=640 plots=True

In [ ]:
!ls {HOME}/runs/detect/train/

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/train/results.png', width=600)

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/train/val_batch0_pred.jpg', width=600)

In [ ]:
!yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

In [ ]:
!yolo task=detect mode=predict model={HOME}/runs/detect/train/weights/best.pt conf=0.25 source={dataset.location}/test/images save=True

In [ ]:
import glob
import os
from IPython.display import Image as IPyImage, display

latest_folder = max(glob.glob(f'{HOME}/runs/detect/predict*/'), key=os.path.getmtime)
for img in glob.glob(f'{latest_folder}/*.jpg')[:3]:
    display(IPyImage(filename=img, width=600))
    print("\n")

In [ ]:
# Export the fine-tuned model to TFLite for Android Studio
# Produces float32 and float16 .tflite files inside best_saved_model/
# Note: if ultralytics warns about tflite being deprecated, use format='litert' instead
from ultralytics import YOLO

best_model = YOLO(f'{HOME}/runs/detect/train/weights/best.pt')
best_model.export(format='tflite')

In [ ]:
# Confirm export files exist
import glob
tflite_files = glob.glob(f'{HOME}/runs/detect/train/weights/best_saved_model/**/*.tflite', recursive=True)
print("TFLite files:", tflite_files)

In [ ]:
# Zip training results
!zip -r yolov26n_results.zip /content/runs/detect/train/

In [ ]:
# Zip the exported TFLite model
!zip -r yolov26n_saved_model.zip /content/runs/detect/train/weights/best_saved_model

In [ ]:
from google.colab import files

files.download('yolov26n_results.zip')
files.download('yolov26n_saved_model.zip')